# Spark Interoperability with the Fleet Iceberg V3 Tables

This notebook queries the **Smart Fleet** Snowflake-managed Iceberg V3 tables from Apache Spark
through the Snowflake **Horizon Iceberg REST catalog**, with **access controls enforced by
Snowflake** (the same masking policies apply cross-engine).

It demonstrates:
- Reading the fleet tables created by the `001`–`010` SQL pipeline via the REST catalog
- `variant_get` over the `TELEMETRY_DATA` VARIANT column
- **Enforced governance**: a masking-protected table can't be read by the plain Iceberg catalog
  (Horizon blocks it). Via the Snowflake Fallback Catalog the same `VEHICLE_REGISTRY` query is
  routed through Snowflake and returns full PII as the engineer role, *masked* PII as `FLEET_ANALYST`

## Prerequisites
1. Completed the Snowflake setup (`task demo-up`) so the fleet tables and masking policies exist
2. Apache **Spark 4.0+** (required for VARIANT support) and **Java 17+**
3. **Two named `snow` CLI connections** (in `~/.snowflake/connections.toml` or `config.toml`),
   both pointing at the same Snowflake account:
   - the project's shared **key-pair** connection (set via `CLI_KEYPAIR_CONNECTION_NAME`, the
     same one used by Snowpipe Streaming and notebook deploy) — mints the short-lived JWT for
     the Horizon REST catalog with `snow connection generate-jwt`. `generate-jwt` **requires**
     a connection configured with a private key (no PAT).
   - a **password** connection (set via `SPARK_PASSWORD_CONNECTION_NAME`) — drives the
     `spark-snowflake` connector; the notebook reads `account`, `user`, and `role` from it.
4. The password connection user's password exported for the `spark-snowflake` connector,
   following the snow CLI convention:
   `export SNOWFLAKE_CONNECTIONS_<PASSWORD_CONNECTION_NAME>_PASSWORD="..."`
5. Config is read from environment variables exported from `.env/iceberg.env`

## Configuration

This demo uses **two** named `snow` CLI connections (both for the same Snowflake account):

- **`CLI_KEYPAIR_CONNECTION_NAME`** — the project's single shared key-pair connection (also
  used by streaming and notebook deploy). Mints the Horizon REST catalog JWT via
  `snow connection generate-jwt` (no PAT). `generate-jwt` mandatorily requires a connection
  configured with a private key.
- **`SPARK_PASSWORD_CONNECTION_NAME`** — a password connection used by the `spark-snowflake`
  connector. `account`, `user`, and `role` are resolved from it, and the connector password
  comes from `SNOWFLAKE_CONNECTIONS_<PASSWORD_CONNECTION_NAME>_PASSWORD`.

The Horizon REST catalog URI is derived from the account.

In [1]:
import itertools
import json
import os
import subprocess
import threading
import time
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path

try:
    import tomllib  # Python 3.11+ stdlib
except ModuleNotFoundError:
    import tomli as tomllib  # type: ignore[no-redef]


@contextmanager
def step(label):
    """Print clear START / DONE / FAILED markers with elapsed time so you can
    tell from the Jupyter output when a cell started and whether it finished.
    """
    start = time.perf_counter()
    print(f"\u25b6 START  {label}  [{datetime.now():%H:%M:%S}]", flush=True)
    try:
        yield
    except Exception as exc:
        elapsed = time.perf_counter() - start
        print(f"\u2716 FAILED {label}  after {elapsed:,.1f}s  ({type(exc).__name__}: {exc})", flush=True)
        raise
    else:
        elapsed = time.perf_counter() - start
        print(f"\u2714 DONE   {label}  in {elapsed:,.1f}s", flush=True)


@contextmanager
def spinner(label, interval=0.5):
    """Animated heartbeat on a single line for long, opaque blocking calls.

    Spark's first ``getOrCreate()`` resolves and downloads the Ivy JARs
    (Iceberg runtime, cloud bundle, Snowflake JDBC + connector), which can take
    minutes with little Python-side feedback. A daemon thread ticks a spinner +
    elapsed seconds until the block exits, so you know it is still working.
    (Spark's own Ivy resolution logs may also print here on the first run.)
    """
    done = threading.Event()
    start = time.perf_counter()
    frames = itertools.cycle("|/-\\")

    def _spin():
        while not done.wait(interval):
            elapsed = time.perf_counter() - start
            print(f"\r  {next(frames)} {label}  ({elapsed:,.0f}s)   ", end="", flush=True)

    thread = threading.Thread(target=_spin, daemon=True)
    thread.start()
    try:
        yield
    finally:
        done.set()
        thread.join()
        elapsed = time.perf_counter() - start
        print(f"\r  \u2714 {label}  (done in {elapsed:,.1f}s)" + " " * 12, flush=True)


def _load_named_connection(name: str) -> dict:
    """Return the params for the named ``snow`` CLI connection.

    Search order:
      1. ``~/.snowflake/connections.toml``  (each connection is a top-level table)
      2. ``~/.snowflake/config.toml``       (each connection under ``[connections.<name>]``)

    Raises ``RuntimeError`` if neither file contains the named connection.
    """
    home = Path.home() / ".snowflake"
    candidates = [home / "connections.toml", home / "config.toml"]
    for path in candidates:
        if not path.exists():
            continue
        with path.open("rb") as fh:
            data = tomllib.load(fh)
        # Layout 1 (connections.toml): [<name>]  <-- top-level table
        if name in data and isinstance(data[name], dict):
            return dict(data[name])
        # Layout 2 (config.toml): [connections.<name>]
        connections = data.get("connections", {})
        if name in connections:
            return dict(connections[name])
    raise RuntimeError(
        f"Connection '{name}' not found in {candidates[0]} or {candidates[1]}"
    )


def generate_jwt(connection: str) -> str:
    """Mint a key-pair JWT for the named snow CLI connection (valid ~60 min).

    Shells out to `snow connection generate-jwt`, which signs a JWT with the
    private key already configured in the connection. No PAT required.
    """
    result = subprocess.run(
        ["snow", "connection", "generate-jwt", "--connection", connection, "--silent"],
        capture_output=True,
        text=True,
        check=True,
    )
    token = result.stdout.strip()
    if not token:
        raise RuntimeError("snow connection generate-jwt returned empty output")
    return token


def exchange_jwt_for_access_token(jwt: str, role: str) -> str:
    """Exchange a key-pair JWT for a Horizon REST catalog access token.

    Snowflake's Horizon (Polaris) catalog expects an OAuth2 client_credentials
    exchange: the JWT is sent as ``client_secret`` and the Snowflake role is
    requested via ``scope = session:role:<role>``. The returned bearer
    ``access_token`` is what Spark passes as ``...catalog.horizon.token``.

    We do the exchange here (rather than letting Iceberg's ``credential``
    handling do it) so the wire format matches Snowflake exactly and the role
    is baked into the token. Token endpoint: ``<catalog_uri>/v1/oauth/tokens``.

    The POST is made with ``curl`` (not urllib/requests) on purpose: it uses
    the OS trust store, so a corporate TLS-inspection proxy's private root CA
    (trusted by the macOS keychain and by ``snow``/``curl``) is honored. The
    bundled Python certifi CA store does not include it, so urllib/requests
    fail with 'self-signed certificate in certificate chain'.
    """
    token_endpoint = f"{horizon_catalog_uri}/v1/oauth/tokens"
    result = subprocess.run(
        [
            "curl", "-sS", "-X", "POST", token_endpoint,
            "--header", "Content-Type: application/x-www-form-urlencoded",
            "--data-urlencode", "grant_type=client_credentials",
            "--data-urlencode", f"scope=session:role:{role}",
            "--data-urlencode", f"client_secret={jwt}",
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    try:
        payload = json.loads(result.stdout)
    except json.JSONDecodeError:
        raise RuntimeError(f"Token exchange returned non-JSON: {result.stdout[:300]}")
    access_token = payload.get("access_token")
    if not access_token:
        hint = ""
        if payload.get("error") == "unauthorized_client":
            hint = (
                f"\n  -> The token's user cannot assume role '{role}'. Grant it:\n"
                f"     GRANT ROLE {role} TO USER <your_user>;\n"
                f"     (a just-granted role can take a moment to take effect — retry.)"
            )
        raise RuntimeError(f"Token exchange (scope=session:role:{role}) returned no access_token: {payload}{hint}")
    return access_token


# --- Password connection -> spark-snowflake connector (account / user / role) ---
# Drives account, user, role and the connector password. These are read from
# this connection in ~/.snowflake/connections.toml or ~/.snowflake/config.toml.
# This connection uses password auth.
password_connection = os.environ["SPARK_PASSWORD_CONNECTION_NAME"]
_conn = _load_named_connection(password_connection)

account = _conn["account"]
user = _conn["user"]
# Connection's default role; used as the connector role unless build_spark()
# is called with an explicit role (the masking demo passes engineer/analyst).
connection_role = _conn.get("role")

# Password for the spark-snowflake connector, from the password connection.
# Follows the snow CLI env-var override convention:
#   export SNOWFLAKE_CONNECTIONS_<PASSWORD_CONNECTION_NAME>_PASSWORD="..."
_pw_var = f"SNOWFLAKE_CONNECTIONS_{password_connection.upper()}_PASSWORD"
try:
    snowflake_password = os.environ[_pw_var]
except KeyError:
    raise RuntimeError(
        f"Password env var {_pw_var} is not set. Export it, e.g.:\n"
        f'    export {_pw_var}="<your_password>"'
    )

# --- Shared key-pair connection -> Horizon REST catalog JWT ---
# `snow connection generate-jwt` REQUIRES a connection configured with a private
# key, so the catalog JWT is minted from the project's single shared key-pair
# connection (CLI_KEYPAIR_CONNECTION_NAME) — the same one used by Snowpipe
# Streaming and notebook deploy. It must target the same Snowflake account as the
# password connection above; only the auth method differs.
keypair_connection = os.environ["CLI_KEYPAIR_CONNECTION_NAME"]

# --- Horizon REST catalog (derived from the connection account) ---
# Format: https://<account_identifier>.snowflakecomputing.com/polaris/api/catalog
# Account identifiers can contain underscores (e.g. ORG-my_account), but DNS
# hostnames may only use letters/digits/hyphens (LDH). Snowflake's rule is to
# replace underscores with hyphens in the URL host; otherwise Java's TLS SNI
# rejects the name ('Contains non-LDH ASCII characters') and the handshake fails.
sf_url = f"{account.replace('_', '-')}.snowflakecomputing.com"
horizon_catalog_uri = f"https://{sf_url}/polaris/api/catalog"

# The Snowflake database (Iceberg catalog/warehouse name).
database_name = os.environ["DEMO_DATABASE_NAME"]

# Bronze schema (Iceberg namespace) holding the fleet tables.
bronze_schema = os.environ.get("DEMO_SCHEMA_NAME_BRONZE", "RAW")

# Warehouse for the spark-snowflake connector.
warehouse = os.environ.get("DEMO_WAREHOUSE_NAME", "FLEET_ANALYTICS_WH")

# Roles for the enforced-masking demo.
analyst_role = os.environ.get("DEMO_ANALYST_ROLE_NAME", "FLEET_ANALYST")
engineer_role = os.environ.get("DEMO_ENGINEER_ROLE_NAME", "V3_DEMO_ICEBERG_ENGINEER_ROLE")

# Cloud provider selects the Iceberg cloud SDK bundle + FileIO. Even with
# Snowflake-managed storage, Spark reads data files through cloud SDK bundles,
# so match this to your Snowflake account's cloud (aws | gcp | azure).
cloud_provider = os.environ.get("SPARK_CLOUD_PROVIDER", "aws").lower()
aws_region = os.environ.get("AWS_REGION", "us-east-1")

# Iceberg runtime + cloud bundle version (Spark 4.0 / Scala 2.13).
iceberg_version = os.environ.get("SPARK_ICEBERG_VERSION", "1.10.1")
scala_version = "2.13"

# spark-snowflake connector + JDBC driver versions.
snowflake_jdbc_version = "3.24.0"
snowflake_spark_connector_version = "3.1.6"

with step("Print Initial Output"):
    print(f"    Keypair (JWT) connection:    {keypair_connection}")
    print(f"    Password (connector) conn:   {password_connection}")
    print(f"    Account:                     {account}")
    print(f"    User:                        {user}")
    print(f"    Connection role:             {connection_role}")
    print(f"    Catalog URI:                 {horizon_catalog_uri}")
    print(f"    Snowflake Database:          {database_name}")
    print(f"    Bronze schema:               {bronze_schema}")
    print(f"    Warehouse:                   {warehouse}")
    print(f"    Analyst role:                {analyst_role}")
    print(f"    Engineer role:               {engineer_role}")
    print(f"    Cloud provider:              {cloud_provider}")
    print(f"    AWS region:                  {aws_region}")
    print(f"    Iceberg version:             {iceberg_version}")

▶ START  Print Initial Output  [09:53:08]
    Keypair (JWT) connection:    demo_dgillis_keypair_auth
    Password (connector) conn:   demo_dgillis_password_auth
    Account:                     sfsenorthamerica-dgillis_aws_useast1_v1
    User:                        dgillis
    Connection role:             accountadmin
    Catalog URI:                 https://sfsenorthamerica-dgillis-aws-useast1-v1.snowflakecomputing.com/polaris/api/catalog
    Snowflake Database:          FLEET_ANALYTICS_DB
    Bronze schema:               RAW
    Warehouse:                   FLEET_ANALYTICS_WH
    Analyst role:                FLEET_ANALYST
    Engineer role:               FLEET_ENGINEER
    Cloud provider:              aws
    AWS region:                  us-east-1
    Iceberg version:             1.10.1
✔ DONE   Print Initial Output  in 0.0s


In [2]:
# Map cloud provider -> (Iceberg cloud bundle, FileIO implementation).
_CLOUD = {
    "aws": ("org.apache.iceberg:iceberg-aws-bundle", "org.apache.iceberg.aws.s3.S3FileIO"),
    "gcp": ("org.apache.iceberg:iceberg-gcp-bundle", "org.apache.iceberg.gcp.gcs.GCSFileIO"),
    "azure": ("org.apache.iceberg:iceberg-azure-bundle", "org.apache.iceberg.azure.adlsv2.ADLSFileIO"),
}
if cloud_provider not in _CLOUD:
    raise ValueError(f"SPARK_CLOUD_PROVIDER must be one of {list(_CLOUD)}; got {cloud_provider!r}")

bundle_pkg, file_io = _CLOUD[cloud_provider]
bundle = f"{bundle_pkg}:{iceberg_version}"

with step("Print Bundle + FileIO Output"):
    print(f"    Using bundle: {bundle}")
    print(f"    Using FileIO: {file_io}")

▶ START  Print Bundle + FileIO Output  [09:53:13]
    Using bundle: org.apache.iceberg:iceberg-aws-bundle:1.10.1
    Using FileIO: org.apache.iceberg.aws.s3.S3FileIO
✔ DONE   Print Bundle + FileIO Output  in 0.0s


## Create a Spark session scoped to a Snowflake role

`build_spark(role)` creates a Spark session authenticated to the Horizon REST catalog as the
given role. Because Snowflake enforces access controls (including masking policies) based on
this role, switching roles changes what the **same** query returns — the enforcement happens
in Snowflake, not in Spark.

In [3]:
from pyspark.sql import SparkSession


def build_spark(role=None, enforce_policies=False):
    """Create a Spark session authed to the Horizon REST catalog as ``role``.

    The Horizon REST catalog authenticates with a key-pair JWT minted from the
    key-pair connection; the spark-snowflake connector authenticates with the
    password connection's user + password. Snowflake enforces access controls
    based on ``role``, so switching roles changes what the same query returns.

    Notes:
    - driver.host / bindAddress pin Spark to localhost (avoids VPN issues)
    - Spark 4.0 + Iceberg for native VARIANT support
    - The first call resolves/downloads the Ivy JARs (slow once, then cached);
      a spinner shows it is still working during that download.

    enforce_policies=False uses the plain Iceberg ``SparkCatalog``: tables are read
    directly from object storage via vended credentials (no Snowflake compute). This
    CANNOT read tables protected by masking/row-access policies — Horizon returns 403.
    enforce_policies=True swaps in the Snowflake ``SnowflakeFallbackCatalog`` (from the
    spark-snowflake connector), which routes policy-protected tables through a Snowflake
    warehouse so the policy is enforced and masked/filtered data is returned to Spark.
    """
    role = role or connection_role
    if role is None:
        raise ValueError(
            "No role available: pass a role to build_spark() or set 'role' "
            f"on the '{password_connection}' connection."
        )

    # Stop any active session so a role switch actually re-authenticates.
    active = SparkSession.getActiveSession()
    if active is not None:
        active.stop()

    # Key-pair JWT is the Horizon REST catalog credential (no PAT). It MUST be
    # minted from the key-pair connection; generate-jwt requires a private key.
    # We then exchange it for a role-scoped bearer access token (the role is
    # baked into the token via scope=session:role:<role>), which Spark passes
    # as `...horizon.token`. Passing the JWT as `credential` instead makes the
    # Iceberg client run its own client_credentials exchange, which Snowflake
    # rejects with 'unauthorized_client'.
    jwt = generate_jwt(keypair_connection)
    access_token = exchange_jwt_for_access_token(jwt, role)

    # JARs Spark resolves via Ivy. Downloaded once, then cached in ~/.ivy2.
    packages = [
        f"org.apache.iceberg:iceberg-spark-runtime-4.0_{scala_version}:{iceberg_version}",
        bundle,
        f"net.snowflake:snowflake-jdbc:{snowflake_jdbc_version}",
        f"net.snowflake:spark-snowflake_{scala_version}:{snowflake_spark_connector_version}",
    ]
    print("Spark packages to resolve (downloaded once, then cached in ~/.ivy2):")
    for pkg in packages:
        print(f"    - {pkg}")

    # Plain Iceberg catalog reads object storage directly via vended credentials.
    # The fallback catalog (spark-snowflake connector) routes policy-protected
    # tables through a Snowflake warehouse so masking/row-access policies apply.
    catalog_impl = (
        "org.apache.spark.sql.snowflake.catalog.SnowflakeFallbackCatalog"
        if enforce_policies
        else "org.apache.iceberg.spark.SparkCatalog"
    )

    with spinner("Resolving / downloading Spark + Iceberg JARs (first run can take a few minutes)"):
        builder = (
            SparkSession.builder
            .appName("Fleet Analytics - Iceberg V3 Interop")
            .master("local[*]")
            .config("spark.driver.host", "127.0.0.1")
            .config("spark.driver.bindAddress", "127.0.0.1")
            .config("spark.jars.packages", ",".join(packages))
            .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
            .config("spark.sql.catalog.horizon", catalog_impl)
            .config("spark.sql.catalog.horizon.type", "rest")
            .config("spark.sql.catalog.horizon.uri", horizon_catalog_uri)
            .config("spark.sql.catalog.horizon.token", access_token)
            .config("spark.sql.catalog.horizon.rest.auth.type", "oauth2")
            .config("spark.sql.catalog.horizon.oauth2-server-uri", f"{horizon_catalog_uri}/v1/oauth/tokens")
            .config("spark.sql.catalog.horizon.warehouse", database_name)
            .config("spark.sql.catalog.horizon.header.X-Iceberg-Access-Delegation", "vended-credentials")
            .config("spark.sql.defaultCatalog", "horizon")
            .config("spark.snowflake.sfURL", sf_url)
            .config("spark.snowflake.sfUser", user)
            .config("spark.snowflake.sfPassword", snowflake_password)
            .config("spark.snowflake.sfDatabase", database_name)
            .config("spark.snowflake.sfSchema", bronze_schema)
            .config("spark.snowflake.sfRole", role)
            .config("spark.snowflake.sfWarehouse", warehouse)
            .config("spark.sql.iceberg.vectorization.enabled", "false")
        )
        if enforce_policies:
            # SnowflakeFallbackCatalog reads data files through Iceberg's FileIO;
            # set the S3 FileIO explicitly (provided by the iceberg-aws-bundle).
            builder = builder.config("spark.sql.catalog.horizon.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        spark = builder.getOrCreate()

    mode = "SnowflakeFallbackCatalog (policy enforcement)" if enforce_policies else "Iceberg SparkCatalog (direct read)"
    print(f"Spark session created as role '{role}' via {mode} (Spark {spark.version})")
    return spark


print("build_spark(role) is ready. Call build_spark(engineer_role) to start.")

build_spark(role) is ready. Call build_spark(engineer_role) to start.


## 1. Connect as the engineer role (full access) and list the fleet tables

In [4]:
spark = build_spark(engineer_role)

with step("List fleet namespaces and tables"):
    spark.sql("SHOW NAMESPACES").show()
    spark.sql(f"SHOW TABLES IN {bronze_schema}").show(truncate=False)

Spark packages to resolve (downloaded once, then cached in ~/.ivy2):
    - org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1
    - org.apache.iceberg:iceberg-aws-bundle:1.10.1
    - net.snowflake:snowflake-jdbc:3.24.0
    - net.snowflake:spark-snowflake_2.13:3.1.6
  | Resolving / downloading Spark + Iceberg JARs (first run can take a few minutes)  (1s)   

  - Resolving / downloading Spark + Iceberg JARs (first run can take a few minutes)  (2s)   

:: loading settings :: url = jar:file:/Users/dgillis/Documents/dev/github/shirc/.venv-spark/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/dgillis/.ivy2.5.2/cache
The jars for the packages stored in: /Users/dgillis/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
net.snowflake#snowflake-jdbc added as a dependency
net.snowflake#spark-snowflake_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-10b8f1dc-7908-4eb5-a628-bc4ff9d1f318;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.10.1 in central
	found net.snowflake#snowflake-jdbc;3.24.0 in central
	found net.snowflake#spark-snowflake_2.13;3.1.6 in central
	found net.snowflake#snowflake-jdbc;3.24.2 in central
	found org.a

  \ Resolving / downloading Spark + Iceberg JARs (first run can take a few minutes)  (2s)   

26/06/12 09:54:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


  ✔ Resolving / downloading Spark + Iceberg JARs (first run can take a few minutes)  (done in 5.2s)            
Spark session created as role 'FLEET_ENGINEER' via Iceberg SparkCatalog (direct read) (Spark 4.0.2)
▶ START  List fleet namespaces and tables  [09:54:39]
+---------+
|namespace|
+---------+
|ANALYTICS|
|  CURATED|
|      RAW|
+---------+

+---------+------------------------+-----------+
|namespace|tableName               |isTemporary|
+---------+------------------------+-----------+
|RAW      |API_WEATHER_DATA        |false      |
|RAW      |MAINTENANCE_LOGS        |false      |
|RAW      |SENSOR_READINGS         |false      |
|RAW      |VEHICLE_LOCATIONS       |false      |
|RAW      |VEHICLE_REGISTRY        |false      |
|RAW      |VEHICLE_TELEMETRY_STREAM|false      |
+---------+------------------------+-----------+

✔ DONE   List fleet namespaces and tables  in 8.6s


## 2. Query the VARIANT telemetry column with `variant_get`

`VEHICLE_TELEMETRY_STREAM.TELEMETRY_DATA` is a VARIANT (Iceberg V3). Spark 4.0 reads it
natively and `variant_get` extracts nested fields by JSON path.

In [5]:
with step("Query telemetry VARIANT with variant_get"):
    spark.sql(f"""
SELECT
    VEHICLE_ID,
    EVENT_TIMESTAMP,
    variant_get(TELEMETRY_DATA, '$.speed_mph', 'double')                 AS speed_mph,
    variant_get(TELEMETRY_DATA, '$.engine.rpm', 'int')                   AS engine_rpm,
    variant_get(TELEMETRY_DATA, '$.engine.fuel_level_pct', 'double')     AS fuel_level_pct,
    variant_get(TELEMETRY_DATA, '$.diagnostics.check_engine', 'boolean') AS check_engine,
    variant_get(TELEMETRY_DATA, '$.metadata.region', 'string')          AS region
FROM {bronze_schema}.VEHICLE_TELEMETRY_STREAM
ORDER BY EVENT_TIMESTAMP DESC
LIMIT 20
""").show(truncate=False)

▶ START  Query telemetry VARIANT with variant_get  [09:55:16]


26/06/12 09:55:22 WARN CorruptStatistics: Ignoring statistics because created_by is null or empty! See PARQUET-251 and PARQUET-297
[Stage 0:>                                                          (0 + 1) / 1]

+----------+--------------------------+---------+----------+--------------+------------+-----------------+
|VEHICLE_ID|EVENT_TIMESTAMP           |speed_mph|engine_rpm|fuel_level_pct|check_engine|region           |
+----------+--------------------------+---------+----------+--------------+------------+-----------------+
|VH-0022   |2026-06-12 00:34:34.322029|27.0     |1211      |74.3          |false       |Mountain West    |
|VH-0036   |2026-06-12 00:34:34.322007|26.2     |1164      |31.4          |false       |California       |
|VH-0048   |2026-06-12 00:34:34.321984|37.9     |2982      |74.8          |false       |Midwest          |
|VH-0049   |2026-06-12 00:34:34.321961|45.7     |3015      |64.6          |false       |Northeast        |
|VH-0009   |2026-06-12 00:34:34.321937|19.1     |2394      |87.4          |false       |Northeast        |
|VH-0008   |2026-06-12 00:34:34.321906|70.9     |2113      |51.4          |false       |Midwest          |
|VH-0010   |2026-06-12 00:34:34.32187

## 3. Enforced governance — full PII as the engineer role

`VEHICLE_REGISTRY` carries driver PII (`DRIVER_NAME`, `DRIVER_EMAIL`, `DRIVER_PHONE`) protected
by Snowflake masking policies. A masking-protected table **cannot** be read by the plain
Iceberg catalog over vended credentials (Horizon returns `403 Forbidden` — that would bypass
the policy). To read it from Spark we switch to the **Snowflake Fallback Catalog**
(`enforce_policies=True`), which routes the query through a Snowflake warehouse so the policy
is evaluated. The engineer role is exempt from masking, so it sees the raw values.

In [7]:
# VEHICLE_REGISTRY is masking-protected, so the plain Iceberg catalog can't read it
# (Horizon returns 403). Use enforce_policies=True: the fallback catalog routes the
# query through Snowflake as FLEET_ENGINEER, which the policy exempts -> full PII.
spark = build_spark(engineer_role, enforce_policies=True)

with step("Read VEHICLE_REGISTRY PII as engineer role (full)"):
    spark.sql(f"""
SELECT VEHICLE_ID, DRIVER_NAME, DRIVER_EMAIL, DRIVER_PHONE
FROM {bronze_schema}.VEHICLE_REGISTRY
ORDER BY VEHICLE_ID
LIMIT 10
""").show(truncate=False)

RuntimeError: Token exchange returned no access_token: {'error': 'unauthorized_client', 'error_description': 'The client is not authorized', 'error_uri': None}

## 4. Same query as `FLEET_ANALYST` — PII is masked

Reconnect with the fallback catalog as the analyst role and run the **identical** query. The
query is again routed through Snowflake, where the masking policy applies to `FLEET_ANALYST`,
so the PII columns come back masked. Nothing changed in the SQL — only the role.

In [ ]:
# Same fallback catalog, but as FLEET_ANALYST: the masking policy applies, so the
# routed-through-Snowflake query returns masked PII. Only the role changed.
spark = build_spark(analyst_role, enforce_policies=True)

with step("Read VEHICLE_REGISTRY PII as FLEET_ANALYST (masked)"):
    spark.sql(f"""
SELECT VEHICLE_ID, DRIVER_NAME, DRIVER_EMAIL, DRIVER_PHONE
FROM {bronze_schema}.VEHICLE_REGISTRY
ORDER BY VEHICLE_ID
LIMIT 10
""").show(truncate=False)

## Summary

Apache Spark read the **same** Snowflake-managed Iceberg V3 fleet tables two ways through the
Horizon REST catalog:

- **Non-policy tables** (telemetry, sensors) — read **directly** from object storage with
  vended credentials, using Spark compute and no Snowflake warehouse.
- **Policy-protected `VEHICLE_REGISTRY`** — read via the **Snowflake Fallback Catalog**, which
  routes the query through Snowflake so masking policies are enforced: full PII for the
  engineer role, masked PII for `FLEET_ANALYST`.

Governance lives with the data: a masked table can't be bypassed by reading raw files, and the
policy is applied consistently regardless of which engine runs the query.